# 2. AED：注意力、束搜索与 CTC 候选重打分

**研究问题**：AED 如何利用历史 token 与声学帧预测下一个 token？Teacher Forcing 与推理为什么不同？N-best 重打分能改变什么？

**方法**：随机初始化一个最小注意力解码器，随后用可控评分函数验证 Exp004–006。

```text
声学状态 [B,T,D] ──────────────────┐
历史 token [B,U] → Embedding → 因果 Self-Attention
                           └→ Cross-Attention(声学 K/V) → logits [B,U,V]
训练：真实历史 → Cross Entropy；推理：预测历史 → Beam Search
CTC N-best ──→ AED 候选分数 ──→ 加权重排
```

**边界**：无真实语音 Encoder，无端到端训练；Beam Search 和重打分采用教学评分器。对应平台题目：`asr_aed_minimal`、`asr_aed_beam`、`asr_attention_rescore`。
**代码来源**：本仓库 `experiments/ctc_rnnt_aed/run.py`（基础版本提交 `e86b41b`）；Notebook 将其教学函数拆为逐格实验，不映射外部项目源码。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## Exp004：Mini-AED

**原理**：Decoder 的因果自注意力只读取当前位置及以前的 token；交叉注意力用 Decoder 状态作 query、声学状态作 key/value，得到软对齐权重。Teacher Forcing 输入 `<BOS>,y₁,…,yᵤ`，目标是 `y₁,…,yᵤ,<EOS>`。每个位置可并行计算，但推理仍需逐 token 生成。

**形状**：声学状态 `[B,T,D]`；历史 token `[B,U]`；自注意力遮罩 `[U,U]`；交叉注意力权重 `[B,U,T]`；logits `[B,U,V]`。

In [ ]:
class MiniAED(nn.Module):
    def __init__(self, vocab=5, dim=8):
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.self_attention = nn.MultiheadAttention(dim, 2, batch_first=True)
        self.cross_attention = nn.MultiheadAttention(dim, 2, batch_first=True)
        self.output = nn.Linear(dim, vocab)

    def forward(self, acoustic, decoder_input):
        x = self.embed(decoder_input)
        length = x.shape[1]
        mask = torch.triu(torch.ones(length, length, dtype=torch.bool, device=x.device), diagonal=1)
        x, _ = self.self_attention(x, x, x, attn_mask=mask)
        x, weights = self.cross_attention(x, acoustic, acoustic)
        return self.output(x), weights


In [ ]:
torch.manual_seed(7)
model = MiniAED(vocab=5, dim=8)
acoustic = torch.randn(1,4,8)
history = torch.tensor([[1,2,3]])  # BOS + 两个真实 token
target = torch.tensor([[2,3,4]])   # 两个真实 token + EOS
logits, weights = model(acoustic, history)
loss = F.cross_entropy(logits.reshape(-1,5), target.reshape(-1))
changed = history.clone(); changed[0,2] = 4
other, _ = model(acoustic, changed)
assert logits.shape == (1,3,5) and weights.shape == (1,3,4)
assert torch.allclose(logits[:,:2], other[:,:2], atol=1e-6)
assert torch.allclose(weights.sum(-1), torch.ones(1,3), atol=1e-6)
loss.backward()
assert model.output.weight.grad is not None
print('logits',tuple(logits.shape),'交叉注意力',tuple(weights.shape),'loss',loss.detach().item())

## Exp005：AED Beam Search

**原理**：每个候选保存 `(token 序列, 累计 log 概率, 是否 EOS)`。每步扩展未完成候选，完成候选保留但不再扩展；排序后保留束宽个。与训练使用真实历史不同，推理评分函数接受已预测历史。

**形状**：评分器接收含 BOS 的 token 元组，返回 `[V]`；输出最多 `beam_size` 条候选。EOS 与最大生成长度控制停止。

In [ ]:
def aed_beam(score, bos, eos, beam_size, max_tokens):
    beams = [((), 0., False)]
    for _ in range(max_tokens):
        candidates = []
        for sequence, log_score, ended in beams:
            if ended:
                candidates.append((sequence, log_score, True))
                continue
            for token, value in enumerate(score((bos,) + sequence).tolist()):
                candidates.append((sequence + (token,), log_score + value, token == eos))
        beams = sorted(candidates, key=lambda x: (-x[1], x[0]))[:beam_size]
        if all(ended for _, _, ended in beams):
            break
    return beams


In [ ]:
def score(prefix):
    if len(prefix)==1: return torch.tensor([-9.,-1.,-.5,-9.])
    return torch.tensor([-9.,-9.,-9.,0.]) if prefix[-1]==1 else torch.tensor([-9.,-9.,-9.,-10.])
beams = aed_beam(score,bos=0,eos=3,beam_size=2,max_tokens=2)
assert beams[0][0] == (1,3) and beams[0][2]
print('全局最好候选：',beams[0])

## Exp006：CTC + AED Rescoring

**原理**：CTC 先生成候选 `N-best`，AED 对每个候选计算条件 log 概率，按 $S(Y)=\lambda S_{CTC}(Y)+(1-\lambda)S_{AED}(Y)$ 排序。此处使用固定候选和可控 AED 分数，专门检验融合逻辑。

**形状**：输入是 `[(token 元组, CTC log 分数)]`；输出同一候选集合的重排结果。正确文本若不在 N-best 中，重打分无法恢复。

In [ ]:
def rescore(candidates, attention_score, ctc_weight):
    return sorted(((tokens, ctc_weight * ctc + (1 - ctc_weight) * attention_score(tokens))
                   for tokens, ctc in candidates), key=lambda x: (-x[1], x[0]))


In [ ]:
candidates = [((1,),-1.), ((2,),-1.2)]
ranked = rescore(candidates, lambda p: 1. if p==(2,) else -1., ctc_weight=.3)
assert ranked[0][0] == (2,)
assert {p for p,_ in ranked} == {p for p,_ in candidates}
print('CTC 候选：',candidates,'重打分后：',ranked)

**验证依据与适用边界**：未来 token 不影响较早位置输出，证明此例中因果遮罩有效；权重归一化与梯度检查只证明计算链可运行。随机模型的 loss 不代表训练成果。这里没有实现完整 CTC+AED 联合训练，也没有测量 N-best recall。